In [ ]:
# MLB 투수 부상 예측 - 2) DNN(MLP) + Bayesian Optimization
# 불펜/선발 두 역할, 이진분류(부상 O/X)와 3종분류(어깨/팔꿈치 구분) 모두 학습/평가한다.
# rolling-window 데이터는 이미 최근 등판 기록을 평균으로 뭉갠 표 데이터라 경기 순서
# (시계열) 정보가 없다. 그래서 LSTM이 아니라 MLP(다층 퍼셉트론)를 사용한다.
# PCA는 적용하지 않는다 - 양성 표본(어깨/팔꿈치)이 1~2%뿐인 데이터라 부상 신호가
# 분산이 작은(PCA가 잘라내는) 방향에 있을 수 있다는 우려 때문에 상관관계 기반
# feature selection까지만 적용한다.
# 데이터: data/bullpen_dataset.csv, data/starter_dataset.csv (컬럼 설명은 data/data_description.md 참고)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from bayes_opt import BayesianOptimization
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {DEVICE}")

In [ ]:
# 데이터 로드 (XGBoost 노트북과 동일한 csv, 동일한 라벨/분리 규칙)

DATA = {
    "bullpen": pd.read_csv("data/bullpen_dataset.csv"),
    "starter": pd.read_csv("data/starter_dataset.csv"),
}

ID_COLS = ["player_id", "window_end_date", "il_start_date", "injury_class_strict", "days_to_injury", "split"]
CATEGORICAL_COLS = ["p_throws", "birth_country"]


def load_role(role, exclude_other=True, binarize=False):
    df = DATA[role].copy()
    if exclude_other:
        df = df[df["label"] != 3].copy()
    if binarize:
        df["label"] = (df["label"] > 0).astype(int)
    for c in CATEGORICAL_COLS:
        df[c] = df[c].astype("category")
    return {s: df[df["split"] == s].reset_index(drop=True) for s in ("train", "val", "test")}


def numeric_feature_cols(df):
    return [c for c in df.columns if c not in ID_COLS + CATEGORICAL_COLS + ["label"]]


def select_uncorrelated_features(train_df, candidate_cols, threshold=0.9):
    corr = train_df[candidate_cols].corr().abs()
    kept = []
    for col in candidate_cols:
        is_redundant = any(pd.notna(corr.loc[col, k]) and corr.loc[col, k] > threshold for k in kept)
        if not is_redundant:
            kept.append(col)
    print(f"[feature_selection] 후보 {len(candidate_cols)}개 -> 상관관계(>|{threshold}|) 제거 후 {len(kept)}개")
    return kept

In [ ]:
# 입력 전처리: 상관관계 필터 -> train 기준 표준화(z-score). 범주형(투구팔/출신국)은
# 나중에 임베딩 레이어에서 학습하므로 여기서는 정수 인덱스로만 바꿔둔다.

CORR_THRESHOLD = 0.9


class TabularDataset(Dataset):
    def __init__(self, X_num, X_cat, y):
        self.X_num = torch.as_tensor(X_num, dtype=torch.float32)
        self.X_cat = torch.as_tensor(X_cat, dtype=torch.long)
        self.y = torch.as_tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_num[idx], self.X_cat[idx], self.y[idx]


def prepare_data(role, label_mode):
    splits = load_role(role, exclude_other=True, binarize=(label_mode == "binary"))
    all_num_cols = numeric_feature_cols(splits["train"])
    kept_num_cols = select_uncorrelated_features(splits["train"], all_num_cols, threshold=CORR_THRESHOLD)

    p_throws_vocab = {v: i + 1 for i, v in enumerate(sorted(splits["train"]["p_throws"].dropna().unique()))}
    country_vocab = {v: i + 1 for i, v in enumerate(sorted(splits["train"]["birth_country"].dropna().unique()))}

    raw = {}
    for s, df in splits.items():
        raw[s] = {
            "X_num": df[kept_num_cols].astype(np.float32).fillna(0.0).to_numpy(),
            "X_cat": np.stack([
                df["p_throws"].map(p_throws_vocab).fillna(0).astype(np.int64).to_numpy(),
                df["birth_country"].map(country_vocab).fillna(0).astype(np.int64).to_numpy(),
            ], axis=1),
            "y": df["label"].astype(np.int64).to_numpy(),
        }

    mean = raw["train"]["X_num"].mean(axis=0, keepdims=True)
    std = raw["train"]["X_num"].std(axis=0, keepdims=True) + 1e-6
    for s in raw:
        raw[s]["X_num"] = (raw[s]["X_num"] - mean) / std

    datasets = {s: TabularDataset(raw[s]["X_num"], raw[s]["X_cat"], raw[s]["y"]) for s in raw}
    meta = {
        "num_dim": raw["train"]["X_num"].shape[1],
        "p_throws_vocab_size": len(p_throws_vocab) + 1,
        "country_vocab_size": len(country_vocab) + 1,
        "n_classes": 2 if label_mode == "binary" else 3,
    }
    return datasets, meta, raw["train"]["y"]

In [ ]:
# 모델 구조: 임베딩(투구팔 2차원, 출신국 4차원) + 표준화된 숫자변수를 이어붙인 뒤
# [Linear -> BatchNorm -> ELU -> Dropout]을 은닉층 수만큼 반복하는 MLP.
# 은닉층 구성은 하이퍼파라미터 2개(층 수 h, 첫 은닉층 노드 수 n1)만 정하면
# 나머지 층 크기가 등차수열로 자동 결정된다.

EMBED_DIM_P_THROWS = 2
EMBED_DIM_COUNTRY = 4


def determine_hidden_sizes(input_dim, hidden_layer_init, hidden_node_init):
    h = max(int(round(hidden_layer_init)), 1)
    n1 = max(int(round(hidden_node_init)), 1)
    if n1 <= input_dim:
        if n1 % h == 0:
            n_hidden_layer = h
            step = n1 // n_hidden_layer
        else:
            n_hidden_layer = h + 1
            step = n1 // n_hidden_layer
        sizes = [max(n1 - step * i, 1) for i in range(n_hidden_layer)]
    else:
        n_hidden_layer = h
        if n_hidden_layer % 2 == 0:
            increase_layers = max(n_hidden_layer // 2 - 1, 1)
        else:
            increase_layers = max(n_hidden_layer // 2, 1)
        decrease_layers = max(n_hidden_layer - increase_layers, 1)
        step_increase = n1 - input_dim
        max_node = n1 + (increase_layers - 1) * step_increase
        step_decrease = max(max_node // (decrease_layers + 1), 1)
        sizes = [max(n1 + step_increase * i, 1) for i in range(increase_layers)]
        sizes += [max(max_node - step_decrease * (i + 1), 1) for i in range(decrease_layers)]
    return sizes


class TabularMLP(nn.Module):
    def __init__(self, num_dim, p_throws_vocab, country_vocab, hidden_sizes, num_classes=3, dropout=0.3):
        super().__init__()
        self.embed_p_throws = nn.Embedding(p_throws_vocab, EMBED_DIM_P_THROWS, padding_idx=0)
        self.embed_country = nn.Embedding(country_vocab, EMBED_DIM_COUNTRY, padding_idx=0)
        prev = num_dim + EMBED_DIM_P_THROWS + EMBED_DIM_COUNTRY
        layers = []
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.head = nn.Sequential(*layers)

    def forward(self, x_num, x_cat):
        combined = torch.cat([
            x_num, self.embed_p_throws(x_cat[:, 0]), self.embed_country(x_cat[:, 1]),
        ], dim=1)
        return self.head(combined)

In [ ]:
# 평가 함수(클래스가 2개면 이진 AUC, 3개면 macro AUC로 자동 분기)

def _auc_any(y_true, y_proba):
    if y_proba.shape[1] == 2:
        return roc_auc_score(y_true, y_proba[:, 1])
    return roc_auc_score(y_true, y_proba, average="macro", multi_class="ovr")


def eval_auc(model, loader):
    model.eval()
    all_y, all_proba = [], []
    with torch.no_grad():
        for x_num, x_cat, y in loader:
            x_num, x_cat = x_num.to(DEVICE), x_cat.to(DEVICE)
            all_proba.append(torch.softmax(model(x_num, x_cat), dim=1).cpu().numpy())
            all_y.append(y.numpy())
    y_true = np.concatenate(all_y)
    y_proba = np.concatenate(all_proba)
    try:
        return _auc_any(y_true, y_proba)
    except ValueError:
        return 0.0


def evaluate(model, loader, name, label_mode):
    """선행연구 비교 표(Accuracy/AUC/F1)와 바로 맞출 수 있도록 세 지표를 dict로 반환.
    이진분류는 부상(1) 클래스 기준 F1, 3종분류는 macro F1을 쓴다."""
    model.eval()
    all_y, all_pred, all_proba = [], [], []
    with torch.no_grad():
        for x_num, x_cat, y in loader:
            x_num, x_cat = x_num.to(DEVICE), x_cat.to(DEVICE)
            proba = torch.softmax(model(x_num, x_cat), dim=1).cpu().numpy()
            all_y.append(y.numpy())
            all_pred.append(proba.argmax(axis=1))
            all_proba.append(proba)
    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)
    y_proba = np.concatenate(all_proba)
    print(f"\n--- {name} (n={len(y_true):,}) ---")
    print(classification_report(y_true, y_pred, digits=3, zero_division=0))
    print("confusion matrix (행=실제, 열=예측):")
    print(confusion_matrix(y_true, y_pred))
    auc = _auc_any(y_true, y_proba)
    accuracy = accuracy_score(y_true, y_pred)
    if label_mode == "binary":
        f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    else:
        f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"AUC: {auc:.3f}  Accuracy: {accuracy:.3f}  F1: {f1:.3f}")
    return {"auc": auc, "accuracy": accuracy, "f1": f1}

In [ ]:
# 하이퍼파라미터 탐색 범위 (Bayesian Optimization으로 5개 탐색)

PBOUNDS = {
    "learning_rate": (-4.0, -2.0),       # 10^x, log scale
    "hidden_layer_init": (1, 5),         # 은닉층 수(h)
    "hidden_node_init": (8, 128),        # 첫 은닉층 노드 수(n1)
    "batch_size": (128, 1024),
    "dropout": (0.1, 0.5),
}


def run(role, label_mode, n_iter=25, init_points=8, quick_epochs=25, quick_patience=5,
        final_epochs=100, final_patience=10):
    print(f"\n{'=' * 70}\n[DNN:{label_mode}] {role.upper()}  device={DEVICE}\n{'=' * 70}")
    datasets, meta, y_train = prepare_data(role, label_mode)
    print(f"train={len(datasets['train']):,} val={len(datasets['val']):,} test={len(datasets['test']):,}"
          f"  입력차원={meta['num_dim']}")

    class_weights = compute_class_weight("balanced", classes=np.arange(meta["n_classes"]), y=y_train)
    criterion_weight = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

    def objective(learning_rate, hidden_layer_init, hidden_node_init, batch_size, dropout):
        batch_size = int(batch_size)
        hidden_sizes = determine_hidden_sizes(meta["num_dim"], hidden_layer_init, hidden_node_init)
        lr = 10 ** learning_rate

        train_loader = DataLoader(datasets["train"], batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(datasets["val"], batch_size=512)

        model = TabularMLP(meta["num_dim"], meta["p_throws_vocab_size"], meta["country_vocab_size"],
                            hidden_sizes, meta["n_classes"], dropout).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=criterion_weight)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

        best_val_auc, patience_left = -1.0, quick_patience
        for _ in range(quick_epochs):
            model.train()
            for x_num, x_cat, y in train_loader:
                x_num, x_cat, y = x_num.to(DEVICE), x_cat.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(x_num, x_cat), y)
                loss.backward()
                optimizer.step()
            val_auc = eval_auc(model, val_loader)
            if val_auc > best_val_auc:
                best_val_auc, patience_left = val_auc, quick_patience
            else:
                patience_left -= 1
                if patience_left <= 0:
                    break
        return best_val_auc

    optimizer = BayesianOptimization(f=objective, pbounds=PBOUNDS, random_state=42, verbose=0)
    optimizer.maximize(init_points=init_points, n_iter=n_iter)
    best = optimizer.max
    p = best["params"]
    hidden_sizes = determine_hidden_sizes(meta["num_dim"], p["hidden_layer_init"], p["hidden_node_init"])
    batch_size = int(p["batch_size"])
    lr = 10 ** p["learning_rate"]
    print(f"최적 하이퍼파라미터: 은닉층={hidden_sizes}  lr={lr:.2e}  batch={batch_size}  "
          f"dropout={p['dropout']:.2f}  (탐색 중 최고 val AUC={best['target']:.4f})")

    train_loader = DataLoader(datasets["train"], batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(datasets["val"], batch_size=512)
    test_loader = DataLoader(datasets["test"], batch_size=512)

    model = TabularMLP(meta["num_dim"], meta["p_throws_vocab_size"], meta["country_vocab_size"],
                        hidden_sizes, meta["n_classes"], p["dropout"]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=criterion_weight)
    final_optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    best_val_auc, best_state, patience_left = -1.0, None, final_patience
    for epoch in range(1, final_epochs + 1):
        model.train()
        total_loss = 0.0
        for x_num, x_cat, y in train_loader:
            x_num, x_cat, y = x_num.to(DEVICE), x_cat.to(DEVICE), y.to(DEVICE)
            final_optimizer.zero_grad()
            loss = criterion(model(x_num, x_cat), y)
            loss.backward()
            final_optimizer.step()
            total_loss += loss.item() * len(y)

        val_auc = eval_auc(model, val_loader)
        if epoch == 1 or epoch % 10 == 0:
            print(f"epoch {epoch:3d}  train_loss={total_loss / len(datasets['train']):.4f}  val_auc={val_auc:.4f}")
        if val_auc > best_val_auc:
            best_val_auc, best_state, patience_left = val_auc, model.state_dict(), final_patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"early stopping (best val_auc={best_val_auc:.4f})")
                break

    model.load_state_dict(best_state)
    val_metrics = evaluate(model, val_loader, "Validation (최적 하이퍼파라미터)", label_mode)
    test_metrics = evaluate(model, test_loader, "Test (최적 하이퍼파라미터)", label_mode)
    return {
        "role": role, "label_mode": label_mode, "input_dim": meta["num_dim"],
        "hidden_sizes": hidden_sizes, "best_params": p,
        "val_auc": val_metrics["auc"], "test_auc": test_metrics["auc"],
        "test_accuracy": test_metrics["accuracy"], "test_f1": test_metrics["f1"],
    }

In [ ]:
# 실행: 불펜/선발 x 이진분류/3종분류 총 4가지 조합을 전부 학습한다.
# 시간이 오래 걸리면 ROLES/LABEL_MODES를 줄여서 원하는 조합만 실행해도 된다.

ROLES = ["bullpen", "starter"]
LABEL_MODES = ["binary", "3class"]
N_ITER = 25
INIT_POINTS = 8

results = []
for role in ROLES:
    for label_mode in LABEL_MODES:
        results.append(run(role, label_mode, N_ITER, INIT_POINTS))

In [ ]:
# 결과 요약 (선행연구 비교 표와 같은 형식: Accuracy / AUC / F1)

results_df = pd.DataFrame(results)[[
    "role", "label_mode", "val_auc", "test_auc", "test_accuracy", "test_f1",
]]
print(results_df.to_string(index=False))
results_df